# Legacy trajectory gözlemcisi — devam hücreleri (2026-10-11)

`scripts/gsd_guidance_ablations_colab.ipynb` içindeki ilk **6 code hazırlık hücresinden sonra**, aynı hazır GPU runtime'da kullanın. Eski notebook'un sonraki deney hücrelerini çalıştırmayın. Mevcut notebook değiştirilmedi.

Baseline: Legacy B70/M800, spectral1.17/SCD1, beta2, gamma=eta=.01, lambda=.95, normal10/Background30. Point-MAE ağırlıkları/BN/token akışı sabit. Yerelde GPU koşusu yapılmadı. GPU smoke/pilot çağrılarını kullanıcı başlatır. **Run all kullanmayın.** Smoke ve pilot ayrı duraklardır.

Güncel plan: önce tüm15 corruption için280 örnek üzerinde **seed0**. Fayda görülürse kuralı sabitleyip seed1/2 ile tekrarlanabilirliğini ayrıca değerlendireceğiz; sayısal bir terfi eşiği henüz yok.


## 7 - Yeni smoke sessionunu başlat (bu hücreyi bir kez çalıştırın)

Bu tek hücre Drive'ı bağlar, GitHub branch'ini günceller, kaynakları doğrular, yeni timestamp'li session ve indeks manifestlerini oluşturur, komutları preview eder ve yalnızca Gaussian/Background smoke'unu başlatır. Eski session ve ZIP'lere dokunmaz. Smoke sonrası durun; all15 pilotunu başlatmayın. Git, tracked bir değişiklikle çakışırsa durur; reset/clean/stash kullanılmaz.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/3DD-TTA')
DRIVE_RESULTS = Path('/content/drive/MyDrive/thesis/results/legacy_trajectory')
BRANCH = 'gsd-smooth-spectrum'
CONDA = ['conda', 'run', '--no-capture-output', '-n', '3dd_tta_env', 'python']

def run(command):
    print('\n$ ' + ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=REPO, check=True)

run(['git', 'status', '--short'])
run(['git', 'fetch', 'origin', BRANCH])
run(['git', 'switch', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
resolved_branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if resolved_branch != BRANCH:
    raise RuntimeError(f'Expected {BRANCH}, found {resolved_branch}')
print(f'Using {resolved_branch} @ {resolved_commit}')

run(CONDA + ['scripts/legacy_trajectory_colab.py', 'verify'])
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SESSION = REPO / 'result/modelnet40_c/legacy_trajectory' / f'{stamp}_trajectory'
if SESSION.exists():
    raise FileExistsError(f'Preserving existing session: {SESSION}')
Path('/content/legacy_trajectory_session.txt').write_text(str(SESSION))
run(CONDA + ['scripts/legacy_trajectory_colab.py', 'init', '--session', str(SESSION)])
run(CONDA + ['scripts/legacy_trajectory_colab.py', 'preview', '--session', str(SESSION)])
print(f'\nFresh smoke session: {SESSION}', flush=True)
run(CONDA + ['scripts/legacy_trajectory_colab.py', 'smoke', '--session', str(SESSION), '--drive-dir', str(DRIVE_RESULTS)])
print('Smoke finished. Stop here, run the result-pack/download cells, and share the full ZIP for review. Do not run the pilot yet.')


## 10 — Sonuç ZIP'i, Drive yedeği ve indirme (smoke/pilot/hata sonrası)

Her çalıştırma yeni handoff ZIP oluşturur; önceki ZIP/dizinler korunur. Başarısız/kısmi çıktılar dahildir. Yalnızca yüzdeleri değil, ham logits/logları içeren tam ZIP'i paylaşın. Drive: `MyDrive/thesis/results/legacy_trajectory/`. İndirme hücresini tarayıcı indirmesine izin vererek çalıştırın. Tekrar aynı çıktı dizinine koşmayın; kısmi pilotta tamamlanan seed'leri yeniden çalıştırmadan önce eksik kapsamı inceleyelim.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
HANDOFF="${SESSION}_handoff_$(date -u +%Y%m%dT%H%M%SZ).zip"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py pack --session "$SESSION" --output "$HANDOFF"
printf '%s' "$HANDOFF" > /content/legacy_trajectory_handoff.txt
conda run --no-capture-output -n 3dd_tta_env python - <<'PY'
from pathlib import Path
from scripts.legacy_trajectory_colab import backup_zip
path = Path('/content/legacy_trajectory_handoff.txt').read_text()
backup_zip(Path(path), Path('/content/drive/MyDrive/thesis/results/legacy_trajectory'))
print('Verified Drive backup:', Path(path).name)
PY


In [ ]:
from pathlib import Path
from google.colab import files
files.download(Path('/content/legacy_trajectory_handoff.txt').read_text())


## 11 — Pilot: smoke incelemesinden sonra ayrı çalıştırın; ardından DUR

Runner teknik smoke kapısını doğrular; bilimsel incelemenin yerine geçmez. Smoke ile aynı Git commit ve doğrulanmış kaynaklar kullanılmalıdır. Pilot: tüm15, sabit280, yalnızca seed0, orijinal10/30 horizon. Seed1/2 ayrı, koşullu bir sonraki aşamadır. Politikalar aynı trace üzerinde hesaplanır; ek all15 baseline yeniden koşulmaz. Tek sealed pilot_seed0.zip + CPU analysis oluşur. Sonra result-pack cell'i tekrar çalıştırıp indirin/paylaşın. Tam benchmark veya erken durdurma otomatik başlatılmaz.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py pilot --session "$SESSION" --drive-dir /content/drive/MyDrive/thesis/results/legacy_trajectory


## 12 — Bağımsız CPU analyzer (opsiyonel; yeni GPU koşusu gerektirmez)

Runner zaten `pilot_analysis` üretir. Yeniden analiz gerekiyorsa yeni çıktı dizini kullanın. Terminal / minimum entropy / eşit sınıf olasılığı ortalaması, sabit probe1/2/3, corrected/broken, oracle tanısı ve corruption içinde original-index bootstrap kayıtları üretilir. Tek seed ile seed SD null kalır; seed kararlılığı veya genel istatistiksel anlamlılık iddiası kurulmaz. Tie-break: terminal, sonra en erken gözlem. Düşük entropy doğruluk veya temporal kararlılık garantisi değildir; oracle etiket kullanır ve inference politikası değildir. Timesteps bağımsız örnek değildir.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
conda run --no-capture-output -n 3dd_tta_env python scripts/analyze_legacy_trajectory.py "$SESSION/pilot_seed0.zip" --output "${SESSION}_analysis_$(date -u +%Y%m%dT%H%M%SZ)"


Bu aşamada erken çıkış uygulanmaz; hızlanma iddiası yoktur. Senkronize bileşen süreleri ve toplamlar gözlemci/hash/I/O maliyetini içerir; saf inference benchmark'ı değildir. B1 canlı erken çıkış, yalnızca gözlenen prefix'e dayalı kural, aynı B1 full-horizon'a karşı ek probe dahil net latency ve bilinmeyen corruption için horizon sonraki açık konulardır. B70 throughput ayrı raporlanır. PG/BN/token değişikliği, M1800 ve başarısız Background100/1000 tekrarları kapsam dışıdır.
